import sys, os
sys.path.insert(0, os.path.abspath('..'))
# Baseline models

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_absolute_error
from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


from src.features import CraigslistFeatureEngineer


ImportError: cannot import name 'CraigslistFeatureEngineer' from 'src.features' (d:\VS code\Projects\PythonProjects\spbu_AI_technology_practice\project\Craiglist-cars-price-prediction\src\features.py)

In [ ]:
data_path = Path("data/interim/train_filtered.csv")
if not data_path.exists():
    raise FileNotFoundError(f"Не найден файл: {data_path}")

df = pd.read_csv(data_path, index_col=0)

y = df["price"].copy()
X = df.drop(columns=["price"]).copy()

print(f"Data path: {data_path}")
print(f"X shape: {X.shape}, y shape: {y.shape}")


In [ ]:
def categorical_without_description(X_df: pd.DataFrame) -> list[str]:
    cat_cols = X_df.select_dtypes(include=["object", "category"]).columns.tolist()
    return [col for col in cat_cols if col != "description"]


def numeric_columns(X_df: pd.DataFrame) -> list[str]:
    return X_df.select_dtypes(exclude=["object", "category"]).columns.tolist()


def make_pipeline(model):
    feature_encoder = ColumnTransformer(
        transformers=[
            (
                "description_tfidf",
                TfidfVectorizer(
                    max_features=20000,
                    ngram_range=(1, 2),
                    min_df=5,
                ),
                "description",
            ),
            (
                "categorical_ohe",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_without_description,
            ),
            (
                "numeric",
                "passthrough",
                numeric_columns,
            ),
        ],
        remainder="drop",
    )

    return Pipeline(
        steps=[
            ("preprocess", CraigslistFeatureEngineer()),
            ("encode", feature_encoder),
            ("model", model),
        ]
    )


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"mae": make_scorer(mean_absolute_error, greater_is_better=False)}

models = {
    "DummyRegressor": DummyRegressor(strategy="median"),
    "LinearRegression": LinearRegression(),
}

rows = []
for name, model in models.items():
    pipe = make_pipeline(model)
    scores = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )

    mae_scores = -scores["test_mae"]
    rows.append(
        {
            "model": name,
            "mae_mean": mae_scores.mean(),
            "mae_std": mae_scores.std(),
            "mae_folds": mae_scores,
        }
    )

results = pd.DataFrame(rows).sort_values("mae_mean").reset_index(drop=True)
results


In [ ]:
best = results.loc[0]
print(f"Best baseline: {best['model']}")
print(f"MAE: {best['mae_mean']:.2f} +/- {best['mae_std']:.2f}")
